## Libraries

In [1]:
# Prevent OpenMP runtime conflict between libraries like PyTorch, TensorFlow, and NumPy on Windows
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
import os
sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchmetrics.classification import MulticlassAccuracy

from tqdm import tqdm
import numpy as np

from danflow import (Trainer,
                    ModelChecker,
                    LearningRateSelector,
                    SmallGrid
                    )

## Load Data

In [2]:
data = torch.load('../saved_values/shuttle_data.pt', weights_only=False)

x_train = data["x_train"]
y_train = data["y_train"]

x_valid = data["x_valid"]
y_valid = data["y_valid"]

x_test = data["x_test"]
y_test = data["y_test"]

## Convert to Tensors

In [3]:
x_train = torch.tensor(np.asarray(x_train), dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)

x_valid = torch.tensor(np.asarray(x_valid), dtype=torch.float32)
y_valid = torch.tensor(np.asarray(y_valid), dtype=torch.long)

x_test = torch.tensor(np.asarray(x_test), dtype=torch.float32)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

## Data Loader

In [4]:
train_dataset = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

valid_dataset = TensorDataset(x_valid, y_valid)
valid_loader = DataLoader(valid_dataset, batch_size=256)

test_dataset = TensorDataset(x_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=256)

## MLP Model

The model is a 3-layer MLP with two hidden layers containing 64 and 32 neurons, respectively, using ReLU activation functions. The output layer contains 7 neurons, corresponding to the 7 target classes.

In [5]:
def mlp_model():
    "Initializes multi layer perceptron model"
    in_features = 9
    num_class = 7
    h1 = 64
    h2 = 32

    model = nn.Sequential(nn.Linear(in_features, h1),
                           nn.ReLU(),
                           nn.Linear(h1, h2),
                           nn.ReLU(),
                           nn.Linear(h2, num_class))

    return model


model = mlp_model()
model

Sequential(
  (0): Linear(in_features=9, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=7, bias=True)
)

## Cross-Entropy Loss
$$
\mathcal{L} = -\log(\hat{y}_{\text{true}})
$$

In [6]:
loss_fn = nn.CrossEntropyLoss()

## Optimizer

In [7]:
optimizer = optim.SGD(model.parameters(),
                      lr=0.01,
                      momentum=0.9,
                      nesterov=True,
                      weight_decay=1e-4)

## Metric

In [8]:
accuracy = MulticlassAccuracy(
    num_classes=7
)

## Model Verification

### Step 1: Check Forward Path

Calculate loss for one batch

In [9]:
checker = ModelChecker(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
)

checker.forward_check(
    train_loader,
    expected_output_size=7,
)

Input shape:  (128, 9)
Target shape: (128,)
Output shape: (128, 7)
Average initial loss (5 batches): 1.8786


ForwardCheckResult(num_batches=5, average_loss=1.8785738945007324, input_shape=(128, 9), target_shape=(128,), output_shape=(128, 7))

### Step 2: Check Backward Path

Select random batches and overfit the model

In [10]:
checker.backward_check(
    train_dataset=train_dataset,
    metric=accuracy,
    target_metric=0.90,
    target_loss=0.01,
    epochs=300
)

Backward check:   0%|          | 0/300 [00:00<?, ?epoch/s]


Initial loss: 1.8613
Final loss:   0.0270
Final metric: 0.6667
Result: The model did not reach the requested overfitting target.


BackwardCheckResult(initial_loss=1.8612787008285523, final_loss=0.026993904262781143, final_metric=0.6666666865348816, epochs_trained=300, target_loss=0.01, target_metric=0.9, success=False, automatic_extension_used=False)

In [11]:
checker.continue_backward(500)

Backward check:   0%|          | 0/500 [00:00<?, ?epoch/s]


Initial loss: 1.8613
Final loss:   0.0090
Final metric: 0.7667
Result: The model did not reach the requested overfitting target.


BackwardCheckResult(initial_loss=1.8612787008285523, final_loss=0.008952585607767105, final_metric=0.7666667103767395, epochs_trained=800, target_loss=0.01, target_metric=0.9, success=False, automatic_extension_used=False)

## Learning Rate Selection

In [13]:
model = mlp_model()


lr_selector = LearningRateSelector(
    model=model,
    optimizer_cls=optim.SGD,
    loss_fn=loss_fn,
    metric=accuracy,
    learning_rates=[0.1, 0.01, 0.001, 0.0001],
    weight_decay=1e-4,
    epochs=5,
)


results = lr_selector.search(train_loader)

LR=0.1


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.52s/batch, metric=0.5157, loss=0.0508]



LR=0.01


Epoch 4: 100%|██████████| 1/1 [00:06<00:00,  6.16s/batch, metric=0.3917, loss=0.2353]



LR=0.001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.70s/batch, metric=0.1429, loss=1.1104]



LR=0.0001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.31s/batch, metric=0.0782, loss=1.8641]



Final Results
+---------------+--------+--------+
| Learning Rate | Metric |  Loss  |
+---------------+--------+--------+
|      0.1      | 0.5157 | 0.0508 |
|      0.01     | 0.3917 | 0.2353 |
|     0.001     | 0.1429 | 1.1104 |
|     0.0001    | 0.0782 | 1.8641 |
+---------------+--------+--------+

Best learning rate: 0.1 (Final loss: 0.0508)


## Small Grid

In [14]:
model = mlp_model()

small_grid = SmallGrid(model=model,
    optimizer_cls=optim.SGD,
    loss_fn=loss_fn,
    metric=accuracy,
    learning_rates=[0.1, 0.15, 0.20, 0.25],
    weight_decays=[0.0, 1e-4, 1e-5, 1e-6],
    epochs=5
)

results = small_grid.search(train_loader)

LR=0.1 | WD=0.0


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.67s/batch, metric=0.4248, loss=0.0667]



LR=0.1 | WD=0.0001


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.63s/batch, metric=0.4269, loss=0.0416]



LR=0.1 | WD=1e-05


...


LR=0.25 | WD=1e-05


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.05s/batch, metric=0.3166, loss=0.3628]



LR=0.25 | WD=1e-06


Epoch 4: 100%|██████████| 1/1 [00:01<00:00,  1.23s/batch, metric=0.4258, loss=0.0397]



Final Results:
+---------------+--------------+--------+--------+
| Learning Rate | Weight Decay | Metric |  Loss  |
+---------------+--------------+--------+--------+
|      0.1      |     0.0      | 0.4248 | 0.0667 |
|      0.1      |    0.0001    | 0.4269 | 0.0416 |
|      0.1      |    1e-05     | 0.4265 | 0.0447 |
|      0.1      |    1e-06     | 0.4270 | 0.0437 |
|      0.15     |     0.0      | 0.4261 | 0.0437 |
|      0.15     |    0.0001    | 0.3326 | 0.6117 |
|      0.15     |    1e-05     | 0.4268 | 0.0421 |
|      0.15     |    1e-06     | 0.3769 | 0.5684 |
|      0.2      |     0.0      | 0.4122 | 0.1992 |
|      0.2      |    0.0001    | 0.4240 | 0.0489 |
|      0.2      |    1e-05     | 0.4230 | 0.0568 |
|      0.2      |    1e-06     | 0.4060 | 0.1526 |
|      0.25     |     0.0      | 0.4211 | 0.0723 |
|      0.25     |    0.0001    | 0.1450 | 0.9448 |
|      0.25     |    1e-05     | 0.3166 | 0.3628 |
|      0.25     |    1e-06     | 0.4258 | 0.0397 |
+-------------